In [ ]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math
import random

In [ ]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.
# Five different attack strategies are attempted. All are detected.

In [ ]:
# ── Shared helpers ────────────────────────────────────────────────────────────

QUBIT_SYMBOL = {
    (0, 's'): '0',
    (1, 's'): '1',
    (0, 'd'): '+',
    (1, 'd'): '-',
}

def encode(bit, basis):
    """
    Encode a bit into a qubit using the chosen basis.
      bit=0, basis='s' : |0>  (no gates)
      bit=1, basis='s' : |1>  (X gate)
      bit=0, basis='d' : |+>  (H gate)
      bit=1, basis='d' : |->  (X then H)
    """
    qc = QuantumCircuit(1, 1)
    if bit == 1:
        qc.x(0)
    if basis == 'd':
        qc.h(0)
    return qc

def measure(circuit, basis):
    """
    Measure a qubit in the chosen basis.
    Standard basis : measure directly.
    Diagonal basis : apply H first (|+>->|0>, |->->|1>), then measure.
    """
    qc = circuit.copy()
    if basis == 'd':
        qc.h(0)
    qc.measure(0, 0)
    return qc

def run_one(circuit):
    """Simulate a single-shot measurement and return the bit result (int)."""
    backend = BasicSimulator()
    t_qc    = transpile(circuit, backend)
    counts  = backend.run(t_qc, shots=1).result().get_counts()
    return int(list(counts.keys())[0])

def print_row(label, values):
    """Print one labelled row of the BB84 table."""
    print(f"{label:<14}", end=" ")
    for v in values:
        print(f"{str(v):<5}", end="")
    print()

def detect(attack_name, alice_bits, alice_bases, bob_bits, bob_bases, n):
    """
    Sift the key and check for errors.
    Alice and Bob compare half their sifted key publicly to detect the attacker.
    """
    matching  = [i for i in range(n) if alice_bases[i] == bob_bases[i]]
    alice_key = [alice_bits[i] for i in matching]
    bob_key   = [bob_bits[i]   for i in matching]
    error_pos = [i for i in matching if alice_bits[i] != bob_bits[i]]

    n_check      = max(1, len(alice_key) // 2)
    errors_found = sum(alice_key[i] != bob_key[i] for i in range(n_check))
    error_rate   = errors_found / n_check

    print(f"  Sifted key length  : {len(alice_key)} bits")
    print(f"  Errors in key      : {len(error_pos)} at positions {error_pos}")
    print(f"  Sample checked     : {n_check} bits")
    print(f"  Errors in sample   : {errors_found}")
    print(f"  Error rate         : {error_rate * 100:.1f}%")
    if error_rate > 0:
        print(f"  >> ATTACKER DETECTED by {attack_name}")
    else:
        print(f"  >> No errors detected in sample (Eve may have been lucky)")
    print()

# ── Alice prepares her qubits (shared across all attacks) ─────────────────────

random.seed(42)
N = 20

alice_bits   = [random.randint(0, 1) for _ in range(N)]
alice_bases  = [random.choice(['s', 'd']) for _ in range(N)]
alice_circuits = [encode(b, bas) for b, bas in zip(alice_bits, alice_bases)]

# Bob's basis choices (same across all attacks for a fair comparison)
bob_bases = [random.choice(['s', 'd']) for _ in range(N)]

print("Alice's bits  :", alice_bits)
print("Alice's bases :", alice_bases)
print("Bob's bases   :", bob_bases)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# ATTACK 1 — Random Intercept-Resend
# ══════════════════════════════════════════════════════════════════════════════
#
# Eve intercepts every qubit and measures in a randomly chosen basis.
# She re-encodes her result and forwards the new qubit to Bob.
#
# Why it fails:
#   When Eve's basis differs from Alice's (50% of the time), she disturbs
#   the qubit. Even if Bob then uses the same basis as Alice, he gets a
#   random result. This causes ~25% errors in the sifted key.

eve1_bases    = [random.choice(['s', 'd']) for _ in range(N)]
eve1_bits     = []
eve1_circuits = []
for qc, e_bas in zip(alice_circuits, eve1_bases):
    bit = run_one(measure(qc, e_bas))
    eve1_bits.append(bit)
    eve1_circuits.append(encode(bit, e_bas))

bob1_bits = [run_one(measure(qc, b)) for qc, b in zip(eve1_circuits, bob_bases)]

print("ATTACK 1 — Random Intercept-Resend")
print("=" * 108)
print_row("Index:",    list(range(N)))
print_row("A bit:",    alice_bits)
print_row("A basis:",  alice_bases)
print_row("A qubit:",  [QUBIT_SYMBOL[(alice_bits[i], alice_bases[i])] for i in range(N)])
print("-" * 108)
print_row("E basis:",  eve1_bases)
print_row("E bit:",    eve1_bits)
print_row("E qubit:",  [QUBIT_SYMBOL[(eve1_bits[i], eve1_bases[i])] for i in range(N)])
print("-" * 108)
print_row("B basis:",  bob_bases)
print_row("B bit:",    [bob1_bits[i] if alice_bases[i] == bob_bases[i] else '?' for i in range(N)])
print_row("match:",    ['<<' if alice_bases[i] == bob_bases[i] else '' for i in range(N)])
print_row("disturbed:",[('ERR' if alice_bases[i] == bob_bases[i] and alice_bits[i] != bob1_bits[i] else '') for i in range(N)])
print("=" * 108)
print()
detect("Attack 1", alice_bits, alice_bases, bob1_bits, bob_bases, N)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# ATTACK 2 — Fixed Standard Basis Intercept-Resend
# ══════════════════════════════════════════════════════════════════════════════
#
# Eve always measures in the standard basis ('s'), hoping Alice also used it.
# For qubits Alice encoded in the diagonal basis, Eve always gets a random
# result and forwards a corrupted qubit to Bob.
#
# Why it fails:
#   Alice uses the diagonal basis ~50% of the time. For all those qubits
#   Eve gets a random result, causing errors whenever Bob also chose the
#   diagonal basis. Expected error rate: ~25%.

eve2_basis    = 's'    # Eve is locked to standard basis
eve2_bits     = []
eve2_circuits = []
for qc in alice_circuits:
    bit = run_one(measure(qc, eve2_basis))
    eve2_bits.append(bit)
    eve2_circuits.append(encode(bit, eve2_basis))

bob2_bits = [run_one(measure(qc, b)) for qc, b in zip(eve2_circuits, bob_bases)]

print("ATTACK 2 — Fixed Standard Basis Intercept-Resend")
print("=" * 108)
print_row("Index:",    list(range(N)))
print_row("A bit:",    alice_bits)
print_row("A basis:",  alice_bases)
print_row("A qubit:",  [QUBIT_SYMBOL[(alice_bits[i], alice_bases[i])] for i in range(N)])
print("-" * 108)
print_row("E basis:",  [eve2_basis] * N)
print_row("E bit:",    eve2_bits)
print_row("E qubit:",  [QUBIT_SYMBOL[(eve2_bits[i], eve2_basis)] for i in range(N)])
print("-" * 108)
print_row("B basis:",  bob_bases)
print_row("B bit:",    [bob2_bits[i] if alice_bases[i] == bob_bases[i] else '?' for i in range(N)])
print_row("match:",    ['<<' if alice_bases[i] == bob_bases[i] else '' for i in range(N)])
print_row("disturbed:",[('ERR' if alice_bases[i] == bob_bases[i] and alice_bits[i] != bob2_bits[i] else '') for i in range(N)])
print("=" * 108)
print()
detect("Attack 2", alice_bits, alice_bases, bob2_bits, bob_bases, N)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# ATTACK 3 — Bit-Flip Attack (X gate)
# ══════════════════════════════════════════════════════════════════════════════
#
# Eve applies the X (NOT) gate to every qubit, attempting to flip all bits,
# then forwards the modified qubit to Bob.
#
# Effect on each basis state:
#   X|0> = |1>   X|1> = |0>   (standard basis: flipped)
#   X|+> = |+>   X|-> = |->   (diagonal basis: unchanged)
#
# Why it fails:
#   The X gate flips every standard-basis qubit, causing 100% errors at
#   positions where Alice used the standard basis. Positions where Alice
#   used the diagonal basis are unaffected. Overall error rate: ~50%.

eve3_circuits = []
for qc in alice_circuits:
    attacked = qc.copy()
    attacked.x(0)       # Eve applies X gate to every qubit
    eve3_circuits.append(attacked)

bob3_bits = [run_one(measure(qc, b)) for qc, b in zip(eve3_circuits, bob_bases)]

print("ATTACK 3 — Bit-Flip Attack (X gate on every qubit)")
print("=" * 108)
print_row("Index:",    list(range(N)))
print_row("A bit:",    alice_bits)
print_row("A basis:",  alice_bases)
print_row("A qubit:",  [QUBIT_SYMBOL[(alice_bits[i], alice_bases[i])] for i in range(N)])
print("-" * 108)
print_row("Eve gate:", ['X'] * N)
print("-" * 108)
print_row("B basis:",  bob_bases)
print_row("B bit:",    [bob3_bits[i] if alice_bases[i] == bob_bases[i] else '?' for i in range(N)])
print_row("match:",    ['<<' if alice_bases[i] == bob_bases[i] else '' for i in range(N)])
print_row("disturbed:",[('ERR' if alice_bases[i] == bob_bases[i] and alice_bits[i] != bob3_bits[i] else '') for i in range(N)])
print("=" * 108)
print()
detect("Attack 3", alice_bits, alice_bases, bob3_bits, bob_bases, N)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# ATTACK 4 — Phase-Flip Attack (Z gate)
# ══════════════════════════════════════════════════════════════════════════════
#
# Eve applies the Z (phase-flip) gate to every qubit.
#
# Effect on each basis state:
#   Z|0> = |0>   Z|1> = |1>   (standard basis: unchanged)
#   Z|+> = |->   Z|-> = |+>   (diagonal basis: flipped)
#
# Why it fails:
#   The Z gate is invisible to standard basis measurements but swaps
#   |+> and |-> in the diagonal basis. This causes 100% errors at
#   positions where Alice used the diagonal basis. Overall error rate: ~50%.

eve4_circuits = []
for qc in alice_circuits:
    attacked = qc.copy()
    attacked.z(0)       # Eve applies Z gate to every qubit
    eve4_circuits.append(attacked)

bob4_bits = [run_one(measure(qc, b)) for qc, b in zip(eve4_circuits, bob_bases)]

print("ATTACK 4 — Phase-Flip Attack (Z gate on every qubit)")
print("=" * 108)
print_row("Index:",    list(range(N)))
print_row("A bit:",    alice_bits)
print_row("A basis:",  alice_bases)
print_row("A qubit:",  [QUBIT_SYMBOL[(alice_bits[i], alice_bases[i])] for i in range(N)])
print("-" * 108)
print_row("Eve gate:", ['Z'] * N)
print("-" * 108)
print_row("B basis:",  bob_bases)
print_row("B bit:",    [bob4_bits[i] if alice_bases[i] == bob_bases[i] else '?' for i in range(N)])
print_row("match:",    ['<<' if alice_bases[i] == bob_bases[i] else '' for i in range(N)])
print_row("disturbed:",[('ERR' if alice_bases[i] == bob_bases[i] and alice_bits[i] != bob4_bits[i] else '') for i in range(N)])
print("=" * 108)
print()
detect("Attack 4", alice_bits, alice_bases, bob4_bits, bob_bases, N)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# ATTACK 5 — Partial Intercept-Resend (50% of qubits)
# ══════════════════════════════════════════════════════════════════════════════
#
# Eve intercepts only half the qubits, hoping a lower rate of disturbance
# will go undetected. Qubits she does not intercept are passed to Bob intact.
#
# Why it fails:
#   Intercepting fewer qubits reduces the error rate (~12.5% instead of 25%)
#   but does not eliminate it. Any non-zero error rate in a noiseless quantum
#   channel is conclusive evidence of an eavesdropper. Increasing the number
#   of check bits makes detection arbitrarily reliable even at low error rates.

intercept_mask = [random.choice([True, False]) for _ in range(N)]
eve5_bases     = [random.choice(['s', 'd']) for _ in range(N)]

eve5_circuits = []
for i, (qc, intercepted) in enumerate(zip(alice_circuits, intercept_mask)):
    if intercepted:
        # Eve intercepts, measures, re-encodes
        bit = run_one(measure(qc, eve5_bases[i]))
        eve5_circuits.append(encode(bit, eve5_bases[i]))
    else:
        # Eve lets the qubit pass through untouched
        eve5_circuits.append(qc)

bob5_bits = [run_one(measure(qc, b)) for qc, b in zip(eve5_circuits, bob_bases)]

print("ATTACK 5 — Partial Intercept-Resend (50% of qubits)")
print("=" * 108)
print_row("Index:",      list(range(N)))
print_row("A bit:",      alice_bits)
print_row("A basis:",    alice_bases)
print_row("A qubit:",    [QUBIT_SYMBOL[(alice_bits[i], alice_bases[i])] for i in range(N)])
print("-" * 108)
print_row("intercepted:",['YES' if intercept_mask[i] else 'pass' for i in range(N)])
print("-" * 108)
print_row("B basis:",    bob_bases)
print_row("B bit:",      [bob5_bits[i] if alice_bases[i] == bob_bases[i] else '?' for i in range(N)])
print_row("match:",      ['<<' if alice_bases[i] == bob_bases[i] else '' for i in range(N)])
print_row("disturbed:",  [('ERR' if alice_bases[i] == bob_bases[i] and alice_bits[i] != bob5_bits[i] else '') for i in range(N)])
print("=" * 108)
print()
detect("Attack 5", alice_bits, alice_bases, bob5_bits, bob_bases, N)

In [ ]:
# ── Summary ───────────────────────────────────────────────────────────────────

def error_rate_summary(a_bits, a_bases, b_bits, b_bases, n):
    matching = [i for i in range(n) if a_bases[i] == b_bases[i]]
    if not matching:
        return 0.0
    errors = sum(a_bits[i] != b_bits[i] for i in matching)
    return errors / len(matching)

attacks = [
    ("Random Intercept-Resend",          bob1_bits, "~25%"),
    ("Fixed Standard Basis IR",           bob2_bits, "~25%"),
    ("Bit-Flip (X gate)",                 bob3_bits, "~50%"),
    ("Phase-Flip (Z gate)",               bob4_bits, "~50%"),
    ("Partial Intercept-Resend (50%)",    bob5_bits, "~12.5%"),
]

print("Summary of All Attacks")
print("=" * 72)
print(f"{'Attack':<35} {'Actual Error':>13} {'Theory':>10} {'Detected?':>10}")
print("-" * 72)
for name, b_bits, theory in attacks:
    er = error_rate_summary(alice_bits, alice_bases, b_bits, bob_bases, N)
    detected = 'YES' if er > 0 else 'NO (lucky)'
    print(f"{name:<35} {er*100:>12.1f}% {theory:>10} {detected:>10}")
print("=" * 72)
print()
print("Conclusion:")
print("  Every attack introduces errors into the sifted key.")
print("  Alice and Bob detect these by publicly comparing a sample of bits.")
print("  No attack can alter or read the key without leaving a detectable trace.")
print("  This is guaranteed by the laws of quantum mechanics, not just algorithm design.")